# Synthetic Tweet Generation — GPT (OpenAI)

Generates synthetic political tweets per `(target, stance)` cell using neutral, label-only prompts. Surface diversity comes from a deterministic 4 × 6 × 4 grid of templates × topics × formats — no political framing is added. Same `SEED_BASE` reproduces the same sequence.

**Form factor:** `max_tokens=70` and a one-line tweet-realism hint in the system prompt keep output close to real-tweet length (~140 chars) instead of essay-style paragraphs.

**Model pinning:** the `MODEL` config uses a *dated* snapshot (e.g., `gpt-4o-mini-2024-07-18`) rather than the floating alias, so future replication points at the same weights.

**Output:** one CSV per run with columns matching P-Stance (`Tweet`, `Target`, `Stance`) plus generation metadata.

In [18]:
import os, re, time, hashlib, random
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## 2. Run configuration

In [19]:
MODEL = "gpt-4o-mini-2024-07-18"   # pinned snapshot for replication (not the floating alias)
TWEETS_PER_CELL = 25      # 25 = pilot, 1200 = full
SEED_BASE       = 42

TARGETS = ["Donald Trump", "Joe Biden", "Bernie Sanders"]
STANCES = ["FAVOR", "AGAINST"]

OUTPUT_DIR = Path("data/synthetic_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model           : {MODEL}")
print(f"Tweets per cell : {TWEETS_PER_CELL}")
print(f"Total cells     : {len(TARGETS) * len(STANCES)}")


Model           : gpt-4o-mini-2024-07-18
Tweets per cell : 25
Total cells     : 6


## 3. Prompt construction

Neutral, label-only prompts. Diversity comes from a deterministic rotation over **4 templates × 6 topics × 4 formats = 96 unique prompts** per cell. Topics and formats are politically symmetric — they shape the surface form of the output, not its political content.

The system prompt includes one neutral form-factor hint ("real tweets are short, often informal, may include hashtags or casual phrasing") to bring synthetic output closer to the form factor of real P-Stance tweets. This is form-factor steering, not content steering — same logic as the existing 280-character cap, just tighter.

In [20]:
SYSTEM_PROMPT = (
    "You are a creative-writing assistant generating fictional social-media "
    "posts for an academic NLP research dataset. The posts are training data "
    "for a stance-classification model — they are not real opinions and will "
    "not be published as real tweets. "
    "Real tweets are short (often 80–160 characters), informal, conversational, "
    "and may include hashtags or casual phrasing. "
    "Output only the post text. No quotes, no preamble, no caveats, no disclaimers."
)

# 4 base templates × 6 topics × 4 formats = 96 unique prompts per cell.
# "short rant" was removed — it encouraged paragraph-style output that didn't match real-tweet form factor.
TEMPLATES = [
    "Write a single tweet that expresses a stance {direction} {target}, focused on {topic}. Format it as {fmt}. Stay under 200 characters.",
    "Compose one tweet from a regular Twitter user whose stance is {direction} {target}. The tweet is about {topic} and reads like {fmt}. Maximum 200 characters.",
    "Generate a realistic tweet expressing a position {direction} {target}. The subject is {topic} and the tone should fit {fmt}. Keep under 200 characters.",
    "Produce a single short tweet — stance: {direction} {target}, topic: {topic}, format: {fmt}. Real Twitter user voice. Under 200 characters.",
]

TOPICS = [
    "their record in office or public life",
    "their position on a policy issue",
    "the campaign trail",
    "a recent news cycle moment",
    "the primary race",
    "the upcoming general election",
]

FORMATS = [
    "a brief personal reaction",
    "a one-line observation",
    "a rhetorical question",
    "a sharp one-liner",
]

def build_prompt(target, stance, idx):
    direction = "in favor of" if stance == "FAVOR" else "against"
    t  = TEMPLATES[idx % len(TEMPLATES)]
    tp = TOPICS[(idx // len(TEMPLATES)) % len(TOPICS)]
    fm = FORMATS[(idx // (len(TEMPLATES) * len(TOPICS))) % len(FORMATS)]
    return t.format(direction=direction, target=target, topic=tp, fmt=fm)

# show the first 6 rotations for one cell — should all look different
for i in range(6):
    print(f"[{i}] {build_prompt('Donald Trump', 'FAVOR', i)}\n")

[0] Write a single tweet that expresses a stance in favor of Donald Trump, focused on their record in office or public life. Format it as a brief personal reaction. Stay under 200 characters.

[1] Compose one tweet from a regular Twitter user whose stance is in favor of Donald Trump. The tweet is about their record in office or public life and reads like a brief personal reaction. Maximum 200 characters.

[2] Generate a realistic tweet expressing a position in favor of Donald Trump. The subject is their record in office or public life and the tone should fit a brief personal reaction. Keep under 200 characters.

[3] Produce a single short tweet — stance: in favor of Donald Trump, topic: their record in office or public life, format: a brief personal reaction. Real Twitter user voice. Under 200 characters.

[4] Write a single tweet that expresses a stance in favor of Donald Trump, focused on their position on a policy issue. Format it as a brief personal reaction. Stay under 200 charact

## 4. Smoke test — verify API access

One synchronous call to confirm the API key and model ID work before submitting a batch. `gpt-4o-mini` is not a reasoning model, so we use `temperature=0.9` for diversity and a small `max_tokens` budget. The `is_refusal` and `normalize` helpers are also defined here for reuse downstream.

In [21]:
REFUSAL_HEADS = ("i can't", "i cannot", "i'm sorry", "as an ai",
                 "i won't", "i'm not able", "i'm unable")

def is_refusal(text):
    if not text or len(text) < 20:
        return True
    return text.lower().lstrip('"\' ').startswith(REFUSAL_HEADS)

def normalize(text):
    return re.sub(r"[^\w\s]", "", text.lower()).strip() if text else ""


In [22]:
# Smoke test — one synchronous call. If this prints non-empty content with
# finish_reason="stop" and length under ~200 chars, the Batch submission below will work too.
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_prompt("Donald Trump", "FAVOR", 0)},
    ],
    temperature=0.9,
    max_tokens=70,
)
choice = resp.choices[0]
content = choice.message.content or ""
print("finish_reason :", choice.finish_reason)
print("length        :", len(content), "chars")
print("content       :", repr(content)[:300])
print("usage         :", resp.usage)


finish_reason : stop
length        : 179 chars
content       : "I've always admired how Trump put America first! His economic policies helped create jobs and boost growth. Proud to support a leader who prioritizes our country! #MAGA #Trump2024"
usage         : CompletionUsage(completion_tokens=37, prompt_tokens=139, total_tokens=176, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


## 5. Generate all tweets

One loop. For each `(target, stance)` cell, keep calling the API until we have `TWEETS_PER_CELL` unique non-refused tweets, with a safety cap at `2× TWEETS_PER_CELL` calls to prevent runaway. Saves a progress CSV after each cell so a crash doesn't lose work.

For full run (`TWEETS_PER_CELL=1200`): ~10–15 minutes total, ~$0.60 at standard pricing.

In [23]:
all_rows = []

for ti, target in enumerate(TARGETS):
    for si, stance in enumerate(STANCES):
        offset    = (ti * len(STANCES) + si) * 100_000
        seen_norm = set()
        kept      = 0
        idx       = 0
        max_tries = TWEETS_PER_CELL * 2

        pbar = tqdm(total=TWEETS_PER_CELL, desc=f"{target}/{stance}", leave=False)
        while kept < TWEETS_PER_CELL and idx < max_tries:
            seed = SEED_BASE + offset + idx
            try:
                resp = client.chat.completions.create(
                    model       = MODEL,
                    messages    = [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": build_prompt(target, stance, idx)},
                    ],
                    temperature = 0.9,
                    max_tokens  = 70,
                    seed        = seed,
                )
                text = (resp.choices[0].message.content or "").strip().strip('"\'')
                err  = None
            except Exception as e:
                text, err = None, str(e)

            refused = is_refusal(text)
            norm    = normalize(text) if text else ""

            # Only count it if it's valid AND new
            if not refused and not err and norm and norm not in seen_norm:
                seen_norm.add(norm)
                all_rows.append({
                    "Tweet": text, "Target": target, "Stance": stance,
                    "model": MODEL, "prompt_idx": idx, "seed": seed,
                    "refused": False, "error": None,
                })
                kept += 1
                pbar.update(1)
            idx += 1
        pbar.close()
        print(f"  {target}/{stance}: kept {kept}/{TWEETS_PER_CELL} after {idx} tries")

        # incremental save in case of crash
        pd.DataFrame(all_rows).to_csv(OUTPUT_DIR / f"{MODEL}_progress.csv", index=False)

# final CSV
ts  = datetime.now().strftime("%Y%m%d_%H%M%S")
df  = pd.DataFrame(all_rows)
out = OUTPUT_DIR / f"{MODEL}_synthetic_{TWEETS_PER_CELL}per_cell_{ts}.csv"
df.to_csv(out, index=False)
print(f"\n✅ Wrote {len(df)} rows → {out}")


  Donald Trump/FAVOR: kept 25/25 after 25 tries


  Donald Trump/AGAINST: kept 25/25 after 30 tries


  Joe Biden/FAVOR: kept 25/25 after 25 tries


  Joe Biden/AGAINST: kept 25/25 after 29 tries


  Bernie Sanders/FAVOR: kept 25/25 after 25 tries


  Bernie Sanders/AGAINST: kept 25/25 after 29 tries

✅ Wrote 150 rows → data\synthetic_data\gpt-4o-mini-2024-07-18_synthetic_25per_cell_20260425_222807.csv


## 7. Output diagnostics

Distribution per Target / per (Target × Stance), refusal rate, sample tweets. Compare the length and hashtag stats below to the real-data EDA in `02_PStanceEDA.ipynb` to spot synthetic-vs-real distribution drift.

In [24]:
print("Final tweets per Target:")
print(df["Target"].value_counts(), "\n")

print("Final tweets per (Target, Stance):")
print(df.groupby(["Target", "Stance"]).size().unstack("Stance").fillna(0).astype(int), "\n")

# Length & hashtag stats — compare with real-data EDA targets
df["char_len"]   = df["Tweet"].str.len()
df["word_len"]   = df["Tweet"].str.split().str.len()
df["n_hashtags"] = df["Tweet"].str.count(r"#\w+")

print("Length + hashtag stats per cell (mean):")
print(df.groupby(["Target", "Stance"])[["char_len", "word_len", "n_hashtags"]].mean().round(1))

# Form-factor parity check vs. real P-Stance baseline (from 02_PStanceEDA)
print(f"\nForm-factor parity vs. real P-Stance:")
print(f"  Synthetic char mean : {df['char_len'].mean():.0f}   (real ≈ 140)")
print(f"  Synthetic hashtags  : {df['n_hashtags'].mean():.2f}/tweet  (real ≈ 1.9)")

print("\nSamples (first 3 of each cell):")
for t in TARGETS:
    for s in STANCES:
        sub = df[(df.Target == t) & (df.Stance == s)]
        print(f"\n[{t} / {s}]")
        for tw in sub["Tweet"].head(3):
            print(f"  • {tw[:200]}")


Final tweets per Target:
Target
Donald Trump      50
Joe Biden         50
Bernie Sanders    50
Name: count, dtype: int64 

Final tweets per (Target, Stance):
Stance          AGAINST  FAVOR
Target                        
Bernie Sanders       25     25
Donald Trump         25     25
Joe Biden            25     25 

Length + hashtag stats per cell (mean):
                        char_len  word_len  n_hashtags
Target         Stance                                 
Bernie Sanders AGAINST     183.9      28.4         1.3
               FAVOR       186.9      29.5         1.7
Donald Trump   AGAINST     179.6      27.2         1.6
               FAVOR       173.3      29.1         1.7
Joe Biden      AGAINST     179.6      27.7         1.6
               FAVOR       183.8      28.6         1.6

Form-factor parity vs. real P-Stance:
  Synthetic char mean : 181   (real ≈ 140)
  Synthetic hashtags  : 1.61/tweet  (real ≈ 1.9)

Samples (first 3 of each cell):

[Donald Trump / FAVOR]
  • I've been imp